# **Modelni moslashtirish**

## **Avval shug'ullantirilgan modelni o'z ma'lumotimizga moslashtirish**

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
!pip install -U datasets fsspec

In [ ]:
!pip install -U transformers

### Ushbu notebookdagi model matnni sentiment tahlil qilishga qaratilgan. Ya'ni, u berilgan matnning ijobiy yoki salbiy kayfiyatda ekanligini aniqlash uchun tayyorlangan.


### **Qisqacha aytganda:**

### Model turi: Bu DistilBERT nomli oldindan o'rgatilgan til modeli bo'lib, u ketma-ketliklarni tasniflash (sequence classification) uchun moslashtirilgan.

### Ma'lumotlar to'plami: Model SST-2 ma'lumotlar to'plamida (The Stanford Sentiment Treebank) qayta o'qitilgan (fine-tuned). Bu to'plam ingliz tilidagi filmlar sharhlarini o'z ichiga oladi va ularni ijobiy yoki salbiy deb belgilaydi.

### Nima bashorat qiladi: Model yangi matnlarni qabul qilib, ularning ijobiy (Ijobiy) yoki salbiy (Salbiy) ekanligini bashorat qiladi. Masalan, "this movie is a masterpiece" jumlasi "Ijobiy" deb, "a complete waste of time" jumlasi esa "Salbiy" deb baholandi.



# **1. Ma’lumotlar to‘plamini yuklash va tayyorlash**





### datasets kutubxonasidan foydalanib, sst2 ma’lumotlar to‘plamini yuklang.




### Modelni tezroq o‘rgatish uchun to‘plamdan kichikroq qismlarni ajratib oling: train uchun 2000 ta va test uchun 400 ta misol.



### seed=42 parametrini ishlating.



In [ ]:
from datasets import load_dataset

dataset = load_dataset("sst2")
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset['test'].shuffle(seed=42).select(range(400))

# **2. Matnni tokenizatsiya qilish**





### distilbert-base-uncased tokenizer’ini yuklang.



### sst2 to‘plamidagi matnlar qisqaroq bo‘lgani uchun max_length=128 qilib, preprocess_function yarating.



### Bu funksiyani small_train va small_test to‘plamlariga map metodi orqali qo‘llang.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    tokenized_inputs = tokenizer(examples["sentence"], truncation=True, padding='max_length', max_length=128)
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)


In [ ]:
print(small_train.column_names)
print(small_test.column_names)

# **3. Modelni shug‘ullantirish**





### distilbert-base-uncased asosida ketma-ketlikni tasniflash uchun model (AutoModelForSequenceClassification) yuklang.



### TrainingArguments sozlamalarini yangi qiymatlar bilan yarating: num_train_epochs=2, logging_steps=50.



### Trainer obyektini yarating va train() metodi bilan modelni shug‘ullantiring.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
import os
import torch

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Modelni aniq CPU ga o'tkazish (Agar yuqoridagi muhit o'zgaruvchisi yetarli bo'lmasa)
# model.to('cpu') kodini faqat CUDA mavjud bo'lganda ishlatish maqsadga muvofiq.
# Agar CUDA_VISIBLE_DEVICES="" o'rnatilgan bo'lsa, PyTorch CUDA ni ko'rmaydi.
# Lekin agar model hali ham GPU da bo'lsa, uni CPU ga o'tkazish kerak.
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

# **4. Modelni baholash va sinovdan o‘tkazish**





### trainer.evaluate() metodini chaqirib, modelning eval_loss ko‘rsatkichini tahlil qiling.



### yangi matnlar ("this movie is a masterpiece", "a complete waste of time") yaratib, modelni shu matnlarda sinab ko‘ring.



### Natijalarni (matn, belgi, ehtimollik) ekranga chiqaring.

In [ ]:
# trainer.evaluate()


In [ ]:
texts = [
    "this movie is a masterpiece", "a complete waste of time"
]

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
import torch
import torch.nn.functional as F

# Inputs ni to'g'ridan-to'g'ri CPU ga yuborish
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

# Modelni ham CPU ga o'tkazilganligiga ishonch hosil qilish
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: "negative", 1: 'possitive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())